# Solution 02: PII Detection and Anonymization

In [1]:
from gliner import GLiNER
import json, os

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "pii_documents.json")) as f:
    pii_documents = json.load(f)

print(f"✓ Loaded {len(pii_documents)} documents")

✓ Loaded 8 documents


## Part 1: Load Model and Detect PII

In [2]:
pii_model = GLiNER.from_pretrained("knowledgator/gliner-pii-edge-v1.0")

test_text = "John Smith called our support line from 415-555-0182. His email is john.smith@example.com."
pii_labels = ["name", "phone number", "email address"]
pii_entities = pii_model.predict_entities(test_text, pii_labels, threshold=0.3)

print(f"Found {len(pii_entities)} entities:")
for e in pii_entities:
    print(f"  [{e['label']:20}] {e['text']!r:30} score={e['score']:.2f}")

/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Found 2 entities:
  [name                ] 'John Smith'                   score=0.91
  [phone number        ] '415-555-0182'                 score=0.92


In [3]:
assert hasattr(pii_model, 'predict_entities')
assert isinstance(pii_entities, list)
assert len(pii_entities) >= 2
labels_found = {e['label'] for e in pii_entities}
assert len(labels_found) >= 2
print(f"✓ Found {len(pii_entities)} PII entities: {[(e['text'], e['label']) for e in pii_entities]}")

✓ Found 2 PII entities: [('John Smith', 'name'), ('415-555-0182', 'phone number')]


## Part 2: Implement `anonymize_text`

In [4]:
def anonymize_text(model, text, labels, threshold=0.3):
    """Replace PII spans in text with <LABEL_TYPE> placeholders."""
    entities = model.predict_entities(text, labels, threshold=threshold)
    # Sort end-to-start to preserve character offsets during replacement
    entities.sort(key=lambda e: e["start"], reverse=True)
    anonymized = text
    for e in entities:
        placeholder = "<" + e["label"].upper().replace(" ", "_") + ">"
        anonymized = anonymized[:e["start"]] + placeholder + anonymized[e["end"]:]
    return anonymized


result = anonymize_text(pii_model, test_text, pii_labels)
print(f"Original:   {test_text}")
print(f"Anonymized: {result}")

assert isinstance(result, str)
assert result != test_text
assert "John Smith" not in result
assert "<" in result and ">" in result
print("✓ anonymize_text works correctly")

Original:   John Smith called our support line from 415-555-0182. His email is john.smith@example.com.
Anonymized: <NAME> called our support line from <PHONE_NUMBER>. His email is john.smith@example.com.
✓ anonymize_text works correctly


In [5]:
hr_text = pii_documents[1]["text"]
hr_labels = ["name", "dob", "ssn", "location address", "phone number"]
hr_result = anonymize_text(pii_model, hr_text, hr_labels)

assert isinstance(hr_result, str)
assert hr_result != hr_text
assert "Sarah Johnson" not in hr_result
print(f"✓ HR document anonymized")
print(f"  Original:   {hr_text}")
print(f"  Anonymized: {hr_result}")

✓ HR document anonymized
  Original:   Employee record: Sarah Johnson, DOB 04/15/1985, SSN 123-45-6789. Home address: 742 Evergreen Terrace, Springfield, IL 62701. Emergency contact: Michael Johnson at 312-555-9988.
  Anonymized: Employee record: <NAME>, DOB <DOB>, SSN <SSN>. Home address: <LOCATION_ADDRESS>. Emergency contact: <NAME> at <PHONE_NUMBER>.


## Part 3: Implement `anonymize_batch`

In [6]:
def anonymize_batch(model, documents, labels, threshold=0.3):
    """Anonymize multiple documents using batch inference."""
    all_entities = model.inference(documents, labels, threshold=threshold)
    results = []
    for text, entities in zip(documents, all_entities):
        entities_sorted = sorted(entities, key=lambda e: e["start"], reverse=True)
        anonymized = text
        for e in entities_sorted:
            placeholder = "<" + e["label"].upper().replace(" ", "_") + ">"
            anonymized = anonymized[:e["start"]] + placeholder + anonymized[e["end"]:]
        results.append(anonymized)
    return results


full_pii_labels = [
    "name", "phone number", "email address", "ssn",
    "credit card", "credit card expiration", "cvv",
    "location address", "dob", "account number",
    "passport number", "driver license", "password", "username",
    "routing number", "bank account", "vehicle id",
]
texts = [doc["text"] for doc in pii_documents]
anonymized = anonymize_batch(pii_model, texts, full_pii_labels)

assert isinstance(anonymized, list)
assert len(anonymized) == len(pii_documents)
for a in anonymized:
    assert isinstance(a, str)
pii_found = sum(1 for a in anonymized if "<" in a)
assert pii_found >= len(pii_documents) // 2

print(f"✓ anonymize_batch processed {len(anonymized)} documents")
print(f"  {pii_found}/{len(pii_documents)} had PII detected")
for i, (orig, anon) in enumerate(zip(texts, anonymized)):
    status = "🔒" if "<" in anon else "  "
    print(f"  {status} Doc {i+1} [{pii_documents[i]['domain']:12}]: {anon[:70]}...")

✓ anonymize_batch processed 8 documents
  8/8 had PII detected
  🔒 Doc 1 [support     ]: Customer <NAME> called our support line from <SSN> on March 10, 2024. ...
  🔒 Doc 2 [hr          ]: Employee record: <NAME>, DOB <DOB>, SSN <SSN>. Home address: <LOCATION...
  🔒 Doc 3 [healthcare  ]: Patient: <NAME>, Date of Birth: <DOB>. Admitted to St. Mary's Medical ...
  🔒 Doc 4 [finance     ]: Transaction alert: A charge of $2,499.00 was made on credit card endin...
  🔒 Doc 5 [insurance   ]: Insurance claim submitted by <NAME>, policy number <DRIVER_LICENSE>. C...
  🔒 Doc 6 [travel      ]: Dear <NAME>, your passport number <PASSPORT_NUMBER> expires on <CREDIT...
  🔒 Doc 7 [it          ]: From: <NAME> <EMAIL_ADDRESS>>
Subject: Login credentials

Hi team, you...
  🔒 Doc 8 [dmv         ]: Driver license renewal notice for <NAME>, DOB <DOB>. License number: <...
